# Topic: Multi-Agent Systems & State Management

## Definition (30-second explanation)
*   A single agent is like a solo freelancer trying to be the CEO, developer, QA tester, and copywriter all at once—eventually, they get overwhelmed and make mistakes.
*   **Multi-Agent Systems** are like a corporate office: tasks are structured as a graph where a **Manager (Router)** delegates work to specialized **Worker Agents** (e.g., a SQL Agent, a Python Agent). 
*   **State Management** is the "shared clipboard" they pass around. Instead of whispering entire conversations down the line, each agent reads the clipboard, updates their specific section, and passes it to the next node.

## Why Interviewers Ask This
*   Single-prompt LLMs hit a ceiling on complex enterprise tasks due to instruction confusion and context limits. 
*   Interviewers want to see if you can architect distributed GenAI workflows using frameworks like LangGraph or AutoGen, and if you know how to govern them safely (preventing infinite loops and token bloat).

## Core Concepts (The 3-Layer Anatomy)
*   **The Bottleneck:** A single agent trying to handle a 10-step process will dilute its attention, forget early instructions, and blow up the context window by holding the entire history of every tool execution.
*   **The Mechanism:** 
    *   **Graph Architecture:** Workflows are modeled as state machines. **Nodes** are Python functions or LLM calls. **Edges** are the routing logic (e.g., `if code_fails -> route to Coder; else -> route to End`).
    *   **State Persistence:** A structured dictionary or Pydantic model (`State`) is passed between nodes. An agent receives the current State, performs an action, and returns a state update (either overwriting a key like `current_code` or appending to a list of `messages`).
*   **The Trade-off:** Multi-agent architectures dramatically increase orchestration complexity, system latency, and API costs. If a Critic agent and a Coder agent disagree, they will burn through dollars arguing with each other.

## When to Use
*   Complex, multi-domain workflows (e.g., AI software engineers like Devin, or comprehensive financial research pipelines).
*   When a task requires multiple distinct personas with different toolsets and system prompts.

## Advantages
*   **Separation of Concerns:** Each agent has a narrow, hyper-focused prompt, drastically reducing hallucinations.
*   **Debugging:** If the pipeline fails, you know exactly *which* node failed and what the state looked like at that exact moment.
*   **Parallelization:** Hierarchical managers can dispatch tasks to 3 worker agents simultaneously, gathering the results when they finish.

## Limitations
*   **Context Blowup:** If agents blindly append every thought and traceback to a shared `messages` list, the token count grows exponentially.
*   **Infinite Loops:** Cyclic graphs (Agent A calls Agent B, who calls Agent A) can run forever if logic is flawed.
*   **Framework Lock-in:** Tools like LangGraph, AutoGen, and CrewAI have steep learning curves and rigid abstractions.

## Common Comparisons
*   **Hierarchical vs. Sequential:** 
    *   *Sequential (Pipeline):* Agent A (Research) $\rightarrow$ Agent B (Draft) $\rightarrow$ Agent C (Publish). Linear and simple.
    *   *Hierarchical (Supervisor):* A Supervisor LLM acts as a router, actively deciding which worker agent to call next based on the state, allowing for dynamic back-and-forth loops.

## Common Interview Traps
*   **Passing the full chat history everywhere:** Candidates often assume every agent needs the entire conversation history. Senior engineers know to pass *only* the relevant state keys (e.g., passing just the `database_schema` to the SQL agent, not the user's conversational pleasantries).
*   **Forgetting the Base Case:** Whenever you have a cycle (e.g., Coder $\leftrightarrow$ Reviewer), you *must* implement a hard stop (e.g., `recursion_limit` or a step counter in the state) to avoid infinite loops.

## Python Syntax (LangGraph Minimal Setup)
```python
from typing import TypedDict
from langgraph.graph import StateGraph, END

# 1. Define the Shared State Schema
class GraphState(TypedDict):
    query: str
    sql_query: str
    error_count: int

# 2. Define Node Functions (The "Agents")
def sql_coder_node(state: GraphState):
    # LLM generates SQL based on state["query"]
    return {"sql_query": "SELECT * FROM sales;", "error_count": state["error_count"]}

def sql_reviewer_node(state: GraphState):
    # Evaluates SQL. If bad, increment error count.
    return {"error_count": state["error_count"] + 1}

# 3. Define Conditional Edge Logic (The "Router")
def route_to_next(state: GraphState):
    if state["error_count"] > 3:
        return "end" # Hard limit prevents infinite loops
    if "ERROR" in state["sql_query"]:
        return "reviewer"
    return "end"

# 4. Build and Compile the Graph
workflow = StateGraph(GraphState)
workflow.add_node("coder", sql_coder_node)
workflow.add_node("reviewer", sql_reviewer_node)

workflow.set_entry_point("coder")
workflow.add_conditional_edges("coder", route_to_next, {"reviewer": "reviewer", "end": END})
workflow.add_edge("reviewer", "coder") # Cycle back

app = workflow.compile()
```

## 45-Second Interview Answer
"Single agents struggle with complex tasks due to prompt dilution and context limits. I build Multi-Agent Systems using a graph-based state machine architecture, like LangGraph. The core is the 'State'—a typed dictionary passed between nodes. Instead of one LLM trying to do everything, I use a Supervisor agent to route the state to specialized worker agents, who perform their tasks and update specific keys in the state. To ensure production stability, I strictly control context window blowup by passing only relevant state keys rather than the full chat history, and I prevent infinite arguing between agents by enforcing hard recursion limits inside the conditional edges."

## Practice Questions:

### Q1: Graph Cycles, Infinite Loops & State Bloat
**Question:** In a cyclic graph where a Coder Agent and a Reviewer Agent pass code back and forth, how do you prevent infinite arguing loops, and how do you prevent the token context from blowing up after multiple failures?

**Answer:**
1. **Preventing Infinite Loops (Recursion Limits):** Cyclic edges (`Coder -> Reviewer -> Coder`) are dangerous. To prevent an infinite loop of repeated hallucinations, I architect a hard cutoff using a Conditional Edge. I add an `error_count: int` to the State schema. Every time the Reviewer rejects the code, it increments the counter. The routing logic explicitly states: `if error_count >= 3: return END_GRAPH`, failing gracefully rather than burning API credits.
2. **Preventing Context Window Blowup (State Key Overwriting):** If agents append every iteration's code and traceback to a shared `messages` list, the prompt grows exponentially. To keep tokens flat, I use **State Key Overwriting**. I define the State with static keys like `current_code: str` and `latest_error: str`. When the Reviewer fails the code, it overwrites `latest_error` with the new traceback. When the Coder tries again, it overwrites `current_code`. The LLM only ever reads the immediate present state, discarding the bulky history of past failures.

**Interview Tips:**
*   **The Overwrite Pattern:** Highlight the difference between *Appending State* (good for chat history) and *Overwriting State* (good for iterative code/data generation).
*   **Graph Governance:** Always emphasize that "loops require limits." Never leave a cyclic edge unmonitored.